In [ ]:
import torch
from ray.rllib.core.rl_module.torch.torch_rl_module import TorchRLModule
import torch.nn as nn

class PiHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(PiHead, self).__init__()
        self._pi_head = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x):
        return self._pi_head(x)
    

class MyTorchPolicy(TorchRLModule):
    def setup(self):
        # You have access here to the following already set attributes:
        # self.observation_space
        # self.action_space
        # self.inference_only
        # self.model_config  # <- a dict with custom settings

        # Use the observation space (if a Box) to infer the input dimension.
        input_dim = self.observation_space.shape[0]

        # Use the model_config dict to extract the hidden dimension.
        hidden_dim = self.model_config["fcnet_hiddens"][0]

        # Use the action space to infer the number of output nodes.
        output_dim = self.action_space.n

        # Build all the layers and subcomponents here you need for the
        # RLModule's forward passes.
        # self._pi_head = torch.nn.Sequential(
        #     torch.nn.Linear(input_dim, hidden_dim),
        #     torch.nn.ReLU(),
        #     torch.nn.Linear(hidden_dim, output_dim),
        # )
        self._pi_head = PiHead(input_dim, hidden_dim, output_dim)

In [1]:
import torch
from ray.rllib.core.rl_module.torch.torch_rl_module import TorchRLModule
import torch.nn as nn

from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.core.rl_module.rl_module import RLModuleSpec

from env.RedBluePebbleGame import RedBluePebbleGameEnv
from env.graph import operator_info, naive_Conv2d_DAG
from model.custom_rllib_module import MyMaskableTorchRLModule, AddActionMaskToBatch

import ray

# Initiate a driver.
ray.init()

op_info = operator_info(
        batch_size=1,
        in_height=7,
        in_width=7,
        inplanes=2,
        kernel_size=3,
        outplanes=2,
        S=12
    )

graph = naive_Conv2d_DAG(op_info)

config = (
    PPOConfig()
    # .environment("CartPole-v1")
    .environment(
        RedBluePebbleGameEnv,
        env_config={
            "verbose": False,
            "summary": False,
            "obs_field": "loc",
            "obs_mode": "vec",
            "history": 1,
            "max_episode_len": 512,
            "reward_config": {
                "TIME_COST": 0,
                "DELETE": 0,
                "LOAD": -1,
                "STORE": -1,
                "COMPUTE": 1,
                "RECOMPUTE": 0,
                "REDUNDANT": 0, # penalty for redundant trap 2023.12.19, masked maybe better
                "DONE": 1       # reward for finish task
            },
            "op_info": op_info,
            "load_lock_enable": True,
            "del_lock_enable": True,
            "action_truncate": False,
        }
    )
    .env_runners(
        # Add custom connector to extract action_mask from info and add to batch
        env_to_module_connector=lambda env, spaces, device: AddActionMaskToBatch(),
        num_env_runners=8,
        num_envs_per_env_runner=2,
        # rollout_fragment_length=256
    )
    .rl_module(
        rl_module_spec=RLModuleSpec(
            module_class=MyMaskableTorchRLModule,
            model_config={
                "addi_features": 0,
                "node_wise": False,
                "edge_index": None,
                "pi_conv_out": 0,
                "vf_conv_out": 0,
                "gp_vf": False,
                "transformer": False,
                "net_arch": {"pi": [1024], "vf": [512]},
                "activation_fn": torch.nn.ReLU,
                "node_num": graph.node_num,
                "mlp_features_extractor_config": {
                    "node_num": graph.node_num,
                    "features_dim": 1024,  # extract output dim(omit the node_num)
                    "node_wise": False,
                    "net_arch": [512],
                    "activation_fn": torch.nn.ReLU,
                },
            },
        ),
    )
    # .training(
    #     # train_batch_size_per_learner=512
    # )
    .learners(
        num_learners=1,
        # num_gpus_per_learner=1
    )
)
ppo = config.build()
# print(ppo.get_module())

from pprint import pprint

try:
    for i in range(3):
        result = ppo.train()
        print(f"env_runners/episode_return_mean: {result['env_runners']['episode_return_mean']}")
    pprint(result)
except Exception as e:
    print("Training error:", e)

/usr/local/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2025-11-30 09:36:12,090	INFO worker.py:1832 -- Connecting to existing Ray cluster at address: 172.16.0.202:6379...
2025-11-30 09:36:12,101	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 
/usr/local/lib/python3.11/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2025-11-30 09:36:12,122	WARNING deprecation.py:50 -- DeprecationWarning: `build` has been deprecated. Use `Algori

ActorDiedError: The actor died because of an error raised in its creation task, [36mray::_WrappedExecutable.__init__()[39m (pid=1875, ip=172.16.0.201, actor_id=3b7574bbcce54bc35da252ed02000000, repr=<ray.train._internal.worker_group._WrappedExecutable object at 0x7fa6ef465c90>)
  At least one of the input arguments for this task could not be computed:
ray.exceptions.RaySystemError: System error: No module named 'env'
traceback: Traceback (most recent call last):
          ^^^^^^^^^^^^^^^^^^^^^^^^^
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
          ^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'env'

In [2]:
from ray import tune

# Create a Tuner instance to manage the trials.
tuner = tune.Tuner(
    config.algo_class,
    param_space=config,
    # Specify a stopping criterion. Note that the criterion has to match one of the
    # pretty printed result metrics from the results returned previously by
    # ``.train()``. Also note that -1100 is not a good episode return for
    # Pendulum-v1, we are using it here to shorten the experiment time.
    run_config=tune.RunConfig(
        stop={"env_runners/episode_return_mean": -1100.0},
    ),
)
# Run the Tuner and capture the results.
results = tuner.fit()

2025-11-29 18:40:35,231	WARNING tune.py:219 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
2025-11-29 18:40:35,235	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/PPO_2025-11-29_18-39-02' in 0.0028s.
2025-11-29 18:40:35,240	INFO tune.py:1041 -- Total run time: 93.20 seconds (93.18 seconds for the tuning loop).
2025-11-29 18:40:35,241	WARNING tune.py:1056 -- Experiment has been interrupted, but the most recent state was saved.
Resume experiment with: Tuner.restore(path="/root/ray_results/PPO_2025-11-29_18-39-02", trainable=...)
2025-11-29 18:40:35,244	WARNING experiment_analysis.py:180 -- Failed to fetch metrics for 1 trial(s):
- PPO_RedBluePebbleGameEnv_9ec50_00000: FileNotFoundError('Could not fetch metrics for PPO_RedBluePebbleGameEnv_9ec50_00000: both result.json and progress.csv w

In [2]:
config.evaluation(
    # Run one evaluation round every iteration.
    evaluation_interval=1,

    # Create 2 eval EnvRunners in the extra EnvRunnerGroup.
    evaluation_num_env_runners=1,

    # Run evaluation for exactly 10 episodes. Note that because you have
    # 2 EnvRunners, each one runs through 5 episodes.
    evaluation_duration_unit="episodes",
    evaluation_duration=1,
)

# Rebuild the PPO, but with the extra evaluation EnvRunnerGroup
ppo_with_evaluation = config.build_algo()

for _ in range(3):
    pprint(ppo_with_evaluation.train())

/usr/local/lib/python3.11/site-packages/ray/rllib/algorithms/algorithm.py:526: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
/usr/local/lib/python3.11/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
The `JsonLogger interface is deprecated in favor of the `ray.tune.json.JsonLoggerCallback` interface and will be removed in Ray 2.7.
  self._loggers.append(cls(self.config, self.logdir, self.trial))
/usr/local/lib/python3.11/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppre

(autoscaler +8m0s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.
(autoscaler +8m0s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.


KeyboardInterrupt: 

(autoscaler +8m35s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.
(autoscaler +9m10s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.
(autoscaler +9m45s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.
(autoscaler +10m20s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.
(autoscaler

In [12]:
import ray
ray.shutdown()

In [13]:
!ray stop

Did not find any active Ray processes.


In [7]:
# ppo = config.build()
print(ppo.get_module())

MyMaskableTorchRLModule(
  (_extract_features): CustomMlp(
    (mlp): Sequential(
      (0): Linear(in_features=4018, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=1024, bias=True)
      (3): ReLU()
    )
  )
  (_mlp_extractor): CustomPolicyValueNet(
    (policy_net): Sequential(
      (0): Linear(in_features=1024, out_features=1024, bias=True)
      (1): ReLU()
    )
    (value_net): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): ReLU()
    )
  )
  (_action_net): Linear(in_features=1024, out_features=1084, bias=True)
  (_value_net): Linear(in_features=512, out_features=1, bias=True)
)


In [6]:
try:
    ppo.train()
except Exception as e:
    print("Training error:", e)

(SingleAgentEnvRunner pid=2615) 2025-11-29 11:16:20,320	ERROR actor_manager.py:186 -- Worker exception caught during `apply()`: 'actions'
(SingleAgentEnvRunner pid=2615) Traceback (most recent call last):
(SingleAgentEnvRunner pid=2615)   File "/usr/local/lib/python3.11/site-packages/ray/rllib/utils/actor_manager.py", line 182, in apply
(SingleAgentEnvRunner pid=2615)     return func(self, *args, **kwargs)
(SingleAgentEnvRunner pid=2615)            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
(SingleAgentEnvRunner pid=2615)   File "/usr/local/lib/python3.11/site-packages/ray/rllib/execution/rollout_ops.py", line 110, in <lambda>
(SingleAgentEnvRunner pid=2615)     else (lambda w: (w.sample(**random_action_kwargs), w.get_metrics()))
(SingleAgentEnvRunner pid=2615)                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(SingleAgentEnvRunner pid=2615)   File "/usr/local/lib/python3.11/site-packages/ray/util/tracing/tracing_helper.py", line 461, in _resume_span
(SingleAgentEnvRunner pid=2615)     return met

KeyboardInterrupt: 